# 09. Statistical Evaluation: TRTR vs TSTR Benchmark

This notebook provides rigorous statistical comparison between:
- **Model A (TRTR)**: Train on Real data, Test on Real data (Baseline)
- **Model B (TSTR)**: Train on Synthetic data, Test on Real data (CVAE-augmented)

## Statistical Design

- **Patient-Level Stratification**: Bootstrap samples patients, not spectra
- **Honest Predictions**: Each patient predicted ONLY by models from its validation fold
- **No Data Leakage**: Fold i models never predict on fold i's patients (they trained on them)
- **Cached Predictions**: Predictions computed once, reused across bootstrap iterations
- **Resampling**: Stratified bootstrap at patient level (cluster bootstrap)
- **Test**: Wilcoxon signed-rank test on bootstrap differences
- **Multiple testing**: Bonferroni correction across 4 models
- **Metrics**: Accuracy, F1-macro, Cohen's κ, McNemar test

## Section 1: Imports & Setup

In [ ]:
import os
import sys
import json
import logging
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import tensorflow as tf
from datetime import datetime
from pathlib import Path
from scipy.stats import binomtest
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    cohen_kappa_score
)

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.utils.config_manager import ConfigManager
from src.notebooks.experiment_utils import setup_experiment
from src.data.loader import GliomaDataLoader

print("✓ Imports successful")

✓ Imports successful


In [ ]:
# Load configuration
config_name = 'exp_101'
config, output_dir = setup_experiment(config_name)

CFG = {
    "data_path": config['paths']['processed_file'],
    "split_file": config['paths']['splits_file'],
    "model_a_dir": config['paths']['models_baseline_dir'],  # TRTR
    "model_b_dir": config['paths']['models_tstr_dir'],      # TSTR
    "output_dir": config['paths']['statistical_dir'],
    "n_bootstrap": config['statistics']['bootstrap_samples'],
    "alpha": config['statistics']['alpha'],
    "random_seed": config['statistics']['random_seed'],
    "models": {
        "Apollo": {"prefix": "apollo", "ext": ".joblib", "type": "sklearn"},
        "RF_Standard": {"prefix": "rf", "ext": ".joblib", "type": "sklearn"},
        "XGBoost": {"prefix": "xgb", "ext": ".json", "type": "xgboost"},
        "1D_CNN": {"prefix": "cnn", "ext": ".keras", "type": "keras"},
    },
    "primary_metric": config['statistics']['primary_metric'],
}

np.random.seed(CFG["random_seed"])
os.makedirs(CFG["output_dir"], exist_ok=True)

print(f"✓ Configuration loaded: {config_name}")
print(f"  Bootstrap samples: {CFG['n_bootstrap']}")
print(f"  α = {CFG['alpha']}")


EXPERIMENT SETUP

Working directory: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data

 Loaded configuration: exp_101
   Description: Complete pipeline: Real data (TRTR) + synthetic (CVAE) + test on synthetic (TSTR) + statistical evaluation

 Output directory: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\exp_101
  Experiment already registered (re-running)

--------------------------------------------------------------------------------
Configuration Summary:
--------------------------------------------------------------------------------
  Preprocessing:
    - AirPLS lambda: 100
    - Signal length: 1738
    - Crop region: None

  CV Strategy:
    - Split source: data/splits/strict_splits.json
    - Use group-aware CV: True
    - Hold out test set: True

  Models:
    - RF n_estimators: 100
    - SVM C: 1.0
    - CVAE epochs: 70


✓ Configuration loaded: exp_101
  Data: data/processed/glioma_dataset.h5
  TRTR models: results/exp_101/models/baselines
  TSTR m

## Section 2: Load Data & Build Patient-to-Fold Mapping

In [ ]:
# Load data
print("Loading data...")
loader = GliomaDataLoader.from_config(
    h5_data_path=CFG['data_path'],
    config=config,
    memory_mapped=False
)

# Load splits from file to determine patient-to-fold mapping
with open(CFG['split_file'], 'r') as f:
    splits_data = json.load(f)

splits = splits_data.get('cv_folds', splits_data)
n_folds = len(splits)

# Collect all validation data and metadata
X_all = []
y_all = []
sample_ids_all = []
patient_ids_all = []
patient_to_fold = {}  # patient_id -> fold_idx (where patient was in validation)

print(f"\nLoading {n_folds} folds...")
loader_iter = iter(loader)
sample_idx = 0

for fold_idx in range(n_folds):
    fold_data = next(loader_iter)
    X_val = fold_data['val'][0].astype(np.float32)
    y_val = fold_data['val'][1].copy()
    
    X_all.append(X_val)
    y_all.append(y_val)
    
    # Get sample IDs for this fold's validation set
    fold_size = X_val.shape[0]
    fold_sample_ids = loader.groups[sample_idx:sample_idx + fold_size]
    sample_ids_all.extend(fold_sample_ids)
    
    # Extract patient IDs
    fold_patient_ids = np.array([s.split('_')[0] + '_' + s.split('_')[1] for s in fold_sample_ids])
    patient_ids_all.extend(fold_patient_ids)
    
    # Build mapping: patient -> fold (this patient's validation fold)
    for pid in np.unique(fold_patient_ids):
        patient_to_fold[pid] = fold_idx
    
    sample_idx += fold_size
    print(f"  Fold {fold_idx}: {fold_size} spectra from {len(np.unique(fold_patient_ids))} patients")

# Combine all folds
X_all = np.vstack(X_all)
y_all = np.concatenate(y_all)
patient_ids_all = np.array(patient_ids_all)
sample_ids_all = np.array(sample_ids_all)

# Fix 1-indexed labels
if np.min(y_all) == 1:
    y_all = y_all - 1

print(f"\n✓ Loaded {X_all.shape[0]} spectra from {len(np.unique(patient_ids_all))} unique patients")
print(f"✓ Patient-to-fold mapping: {len(patient_to_fold)} patients")

Initializing data loader...
Loading CV splits from: data/splits/strict_splits.json
Loading data from data\processed\glioma_dataset.h5...
 Loaded 197596 Total Spectra (Tumor + Healthy).

Loading validation data from 5 folds...

 Using CV splits from: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 131932 train samples, 48118 val samples
  Fold 0: 48118 samples from 10 patients
   Fold 1: 146045 train samples, 34005 val samples
  Fold 1: 34005 samples from 8 patients
   Fold 2: 149039 train samples, 31011 val samples
  Fold 2: 31011 samples from 9 patients
   Fold 3: 136360 train samples, 43690 val samples
  Fold 3: 43690 samples from 17 patients
   Fold 4: 156824 train samples, 23226 val samples
  Fold 4: 23226 samples from 9 patients
✓ Combined test set: 180050 spectra from 49 patient

## Section 3: Helper Functions

In [ ]:
def find_model_path(base_dir: str, prefix: str, ext: str, fold_idx: int) -> str:
    """Find model file for a specific fold."""
    candidates = [
        f"{base_dir}/{prefix}_fold_{fold_idx}{ext}",
        f"{base_dir}/{prefix}_fold{fold_idx}{ext}",
        f"{base_dir}/{prefix}_{fold_idx}{ext}",
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Model '{prefix}' fold {fold_idx} not found")


def load_and_predict(path: str, model_type: str, X: np.ndarray) -> np.ndarray:
    """Load model and return predictions (class labels)."""
    if model_type == "sklearn":
        model = joblib.load(path)
        return model.predict(X).astype(int)
    
    elif model_type == "xgboost":
        model = xgb.XGBClassifier()
        model.load_model(path)
        return model.predict(X).astype(int)
    
    elif model_type == "keras":
        X_3d = X.reshape(-1, X.shape[1], 1)
        model = tf.keras.models.load_model(path, compile=False)
        preds = np.argmax(model.predict(X_3d, verbose=0), axis=1)
        tf.keras.backend.clear_session()
        return preds.astype(int)
    else:
        raise ValueError(f"Unknown model type: '{model_type}'")


def get_fold_specific_predictions(preds_by_fold, patient_ids, patient_to_fold):
    """
    Extract fold-specific predictions (NO DATA LEAKAGE).
    
    For each patient, use ONLY the predictions from the fold where that patient
    was in the validation set (i.e., the fold whose model was trained WITHOUT that patient).
    
    Args:
        preds_by_fold: (n_folds, n_samples) array of predictions from each fold's model
        patient_ids: (n_samples,) array of patient IDs
        patient_to_fold: dict mapping patient_id -> fold_idx (patient's validation fold)
    
    Returns:
        preds: (n_samples,) array where preds[i] comes from the model trained on
               patient_ids[i]'s validation fold
    """
    preds = np.zeros(len(patient_ids), dtype=int)
    
    for i, patient_id in enumerate(patient_ids):
        fold_j = patient_to_fold[patient_id]
        preds[i] = preds_by_fold[fold_j][i]
    
    return preds


def stratified_patient_bootstrap(preds_A, preds_B, y_true, patient_ids, n_boot=1000, seed=42):
    """
    Patient-level stratified bootstrap resampling.
    
    Samples patients (with all their spectra) stratified by class, preserving patient structure.
    """
    rng = np.random.default_rng(seed)
    unique_classes = np.unique(y_true)
    
    boot_diffs = []
    
    # Map each class to its unique patients
    class_to_patients = {}
    for cls in unique_classes:
        class_to_patients[cls] = np.unique(patient_ids[y_true == cls])
    
    for _ in range(n_boot):
        boot_indices = []
        
        # For each class, sample patients with replacement
        for cls in unique_classes:
            patients_in_cls = class_to_patients[cls]
            sampled_patients = rng.choice(patients_in_cls, size=len(patients_in_cls), replace=True)
            
            # Get all spectra for sampled patients
            for patient in sampled_patients:
                patient_mask = (patient_ids == patient) & (y_true == cls)
                patient_indices = np.where(patient_mask)[0]
                boot_indices.extend(patient_indices)
        
        boot_indices = np.array(boot_indices)
        acc_A = accuracy_score(y_true[boot_indices], preds_A[boot_indices])
        acc_B = accuracy_score(y_true[boot_indices], preds_B[boot_indices])
        boot_diffs.append(acc_B - acc_A)
    
    return np.array(boot_diffs)


print("✓ Helper functions defined")

✓ Helper functions defined


## Section 4: Cache Predictions on Disk

In [ ]:
# Create cache directory
cache_dir = Path(CFG['output_dir']) / 'prediction_cache'
cache_dir.mkdir(parents=True, exist_ok=True)

print(f"Loading/caching predictions (one per fold per model)...\n")

# For each fold and each model, load once and cache
cached_preds = {}  # (fold_idx, model_type, model_name) -> predictions

for fold_idx in range(n_folds):
    for model_name, m_cfg in CFG['models'].items():
        for model_type_key in ['A', 'B']:  # A=TRTR, B=TSTR
            model_type_name = 'TRTR' if model_type_key == 'A' else 'TSTR'
            
            # Build cache path
            cache_file = cache_dir / f"preds_fold{fold_idx}_{model_name}_{model_type_name}.npy"
            
            # Check if already cached
            if cache_file.exists():
                preds = np.load(cache_file)
                print(f"  ✓ Loaded cached: Fold {fold_idx} {model_name} {model_type_name}")
            else:
                # Load model and predict
                model_dir = CFG['model_a_dir'] if model_type_key == 'A' else CFG['model_b_dir']
                
                try:
                    path = find_model_path(model_dir, m_cfg['prefix'], m_cfg['ext'], fold_idx)
                    preds = load_and_predict(path, m_cfg['type'], X_all)
                    np.save(cache_file, preds)
                    print(f"  ✓ Computed & cached: Fold {fold_idx} {model_name} {model_type_name}")
                except FileNotFoundError as e:
                    print(f"  ✗ Fold {fold_idx} {model_name} {model_type_name}: Not found")
                    continue
            
            cached_preds[(fold_idx, model_type_key, model_name)] = preds

print(f"\n✓ Predictions cached in: {cache_dir}")

✓ Statistical functions defined


## Section 5: Statistical Evaluation Loop

In [ ]:
results_all = []
alpha_corrected = CFG['alpha'] / len(CFG['models'])

print(f"\n{'='*70}")
print(f"STATISTICAL EVALUATION: TRTR vs TSTR (Patient-Level Bootstrap)")
print(f"{'='*70}")
print(f"Bonferroni-corrected α = {alpha_corrected:.4f}\n")

for model_name in sorted(CFG['models'].keys()):
    print(f"\n{'-'*70}")
    print(f"MODEL: {model_name}")
    print(f"{'-'*70}")
    
    # Get fold-specific predictions for this model
    preds_A_by_fold = []
    preds_B_by_fold = []
    
    for fold_idx in range(n_folds):
        try:
            preds_A = cached_preds[(fold_idx, 'A', model_name)]
            preds_B = cached_preds[(fold_idx, 'B', model_name)]
            preds_A_by_fold.append(preds_A)
            preds_B_by_fold.append(preds_B)
        except KeyError:
            print(f"  ✗ Fold {fold_idx}: Predictions missing")
            continue
    
    if not preds_A_by_fold or not preds_B_by_fold:
        print(f"⚠️  {model_name}: Insufficient predictions. Skipping.")
        continue
    
    # Convert to arrays
    preds_A_by_fold = np.array(preds_A_by_fold)
    preds_B_by_fold = np.array(preds_B_by_fold)
    
    # Extract fold-specific predictions (NO LEAKAGE)\n    preds_A = get_fold_specific_predictions(preds_A_by_fold, patient_ids_all, patient_to_fold)
    preds_B = get_fold_specific_predictions(preds_B_by_fold, patient_ids_all, patient_to_fold)
    
    # Compute baseline metrics
    acc_A = accuracy_score(y_all, preds_A)
    acc_B = accuracy_score(y_all, preds_B)
    f1_A = f1_score(y_all, preds_A, average="macro", zero_division=0)
    f1_B = f1_score(y_all, preds_B, average="macro", zero_division=0)
    kappa_A = cohen_kappa_score(y_all, preds_A)
    kappa_B = cohen_kappa_score(y_all, preds_B)
    
    print(f"\nAggregate Metrics (no leakage):")
    print(f"  TRTR: Acc={acc_A:.4f}  F1={f1_A:.4f}  κ={kappa_A:.4f}")
    print(f"  TSTR: Acc={acc_B:.4f}  F1={f1_B:.4f}  κ={kappa_B:.4f}")
    print(f"  Δ: {acc_B - acc_A:+.4f}")
    
    # Patient-level stratified bootstrap
    print(f"\nRunning patient-level bootstrap ({CFG['n_bootstrap']} iterations)...")
    boot_diffs = stratified_patient_bootstrap(
        preds_A, preds_B, y_all, patient_ids_all,
        n_boot=CFG['n_bootstrap'],
        seed=CFG['random_seed']
    )
    
    # Bootstrap statistics
    mean_diff = np.mean(boot_diffs)
    median_diff = np.median(boot_diffs)
    ci_lo = np.quantile(boot_diffs, 0.025)
    ci_hi = np.quantile(boot_diffs, 0.975)
    
    print(f"\nBootstrap Distribution (Δ = TSTR - TRTR):")
    print(f"  Mean:   {mean_diff:+.4f}")
    print(f"  Median: {median_diff:+.4f}")
    print(f"  95% CI: [{ci_lo:+.4f}, {ci_hi:+.4f}]")
    
    # Wilcoxon test
    n_pos = np.sum(boot_diffs > 0)
    n_nonzero = np.sum(boot_diffs != 0)
    
    if n_nonzero > 0:
        result = binomtest(n_pos, n_nonzero, 0.5, alternative='greater')
        p_wilcoxon = result.pvalue
        z_score = (n_pos - n_nonzero/2) / np.sqrt(n_nonzero/4)
        effect_r = abs(z_score) / np.sqrt(len(y_all))
    else:
        p_wilcoxon = 1.0
        z_score = 0.0
        effect_r = 0.0
    
    sig_nominal = p_wilcoxon < CFG['alpha']
    sig_corrected = p_wilcoxon < alpha_corrected
    
    print(f"\nWilcoxon Test:")
    print(f"  Bootstrap samples with Δ > 0: {n_pos}/{n_nonzero}")
    print(f"  p-value: {p_wilcoxon:.4f}")
    print(f"  Significant (Bonferroni): {'✓' if sig_corrected else '✗'}")
    print(f"  Effect size r: {effect_r:.4f}")
    
    # McNemar test
    correct_A = (preds_A == y_all)
    correct_B = (preds_B == y_all)
    n_11 = np.sum(correct_A & correct_B)
    n_10 = np.sum(correct_A & ~correct_B)
    n_01 = np.sum(~correct_A & correct_B)
    n_00 = np.sum(~correct_A & ~correct_B)
    
    if n_10 + n_01 > 0:
        mcn = mcnemar([[n_11, n_10], [n_01, n_00]], exact=False, correction=True)
        p_mcnemar = mcn.pvalue
    else:
        p_mcnemar = 1.0
    
    print(f"\nMcNemar Test:")
    print(f"  Both correct: {n_11}, Only TRTR: {n_10}, Only TSTR: {n_01}, Both wrong: {n_00}")
    print(f"  p-value: {p_mcnemar:.4f}")
    
    # Store results
    results_all.append({
        "model": model_name,
        "acc_A": float(acc_A),
        "acc_B": float(acc_B),
        "acc_delta": float(acc_B - acc_A),
        "f1_A": float(f1_A),
        "f1_B": float(f1_B),
        "kappa_A": float(kappa_A),
        "kappa_B": float(kappa_B),
        "mean_diff": float(mean_diff),
        "median_diff": float(median_diff),
        "ci_lo": float(ci_lo),
        "ci_hi": float(ci_hi),
        "p_wilcoxon": float(p_wilcoxon),
        "effect_r": float(effect_r),
        "sig_corrected": bool(sig_corrected),
        "p_mcnemar": float(p_mcnemar),
    })

print(f"\n{'='*70}")
print(f"Evaluation complete: {len(results_all)} models")
print(f"{'='*70}")


STATISTICAL EVALUATION: TRTR vs TSTR

Bonferroni-corrected significance level: α = 0.0125 (from 0.05 / 4 models)


──────────────────────────────────────────────────────────────────────
MODEL: 1D_CNN
──────────────────────────────────────────────────────────────────────
Loading fold-trained models and predicting on test data...


## Section 6: Save & Summarize Results

In [ ]:
# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = Path(CFG['output_dir']) / f"statistical_evaluation_{timestamp}.json"

with open(json_path, 'w') as f:
    json.dump(results_all, f, indent=2)

print(f"✓ Results saved: {json_path}")

# Summary table
summary_df = pd.DataFrame([
    {
        "Model": r["model"],
        "Acc(TRTR)": f"{r['acc_A']:.4f}",
        "Acc(TSTR)": f"{r['acc_B']:.4f}",
        "Δ": f"{r['acc_delta']:+.4f}",
        "p-val": f"{r['p_wilcoxon']:.4f}",
        "Sig": "✓" if r['sig_corrected'] else "",
        "Effect": f"{r['effect_r']:.3f}",
    }
    for r in results_all
])

print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
print(summary_df.to_string(index=False))

csv_path = Path(CFG['output_dir']) / f"statistical_summary_{timestamp}.csv"
summary_df.to_csv(csv_path, index=False)
print(f"\n✓ Summary saved: {csv_path}")

✓ Results saved to: results/exp_101/statistical_eval\statistical_evaluation_20260408_110000.json

SUMMARY TABLE

Empty DataFrame
Columns: []
Index: []

✓ Summary saved to: results/exp_101/statistical_eval\statistical_summary_20260408_110000.csv


## Section 7: Interpretation

In [ ]:
print(f"\n{'='*70}")
print("FINDINGS")
print(f"{'='*70}\n")

sig_models = [r for r in results_all if r['sig_corrected']]
improving = [r for r in results_all if r['acc_delta'] > 0]
degrading = [r for r in results_all if r['acc_delta'] < 0]

print(f"Statistically Significant Improvements (α = {alpha_corrected:.4f}):")
if sig_models:
    for r in sig_models:
        print(f"  • {r['model']:12s}: Δ = {r['acc_delta']:+.4f}, p = {r['p_wilcoxon']:.4f}")
else:
    print("  (None)")

print(f"\nImprovement Trend (not significant):")
for r in improving:
    if not r['sig_corrected']:
        print(f"  • {r['model']:12s}: Δ = {r['acc_delta']:+.4f}, p = {r['p_wilcoxon']:.4f}")

if degrading:
    print(f"\nDegradation:")
    for r in degrading:
        print(f"  • {r['model']:12s}: Δ = {r['acc_delta']:+.4f}, p = {r['p_wilcoxon']:.4f}")


FINDINGS & INTERPRETATION



ZeroDivisionError: float division by zero